In [13]:
import pandas as pd
import numpy as np

# processed dataset
df = pd.read_csv("processed_data.csv")

In [14]:
# Load file
import pandas as pd

file_path = 'processed_data.csv'
processed_data = pd.read_csv(file_path)

# Display the columns
print(processed_data.columns)

columns_list = list(processed_data.columns)
print(columns_list)

Index(['FL_DATE', 'CRS_DEP_TIME', 'OP_UNIQUE_CARRIER', 'DEST',
       'wind_dir_degrees', 'wind_speed_kt', 'wind_gust_kt',
       'visibility_statute_mi', 'temperature_c', 'dewpoint_c', 'altimeter_hpa',
       'Flight_Status', 'Carrier_AA', 'Carrier_AS', 'Carrier_B6', 'Carrier_DL',
       'Carrier_F9', 'Carrier_G4', 'Carrier_HA', 'Carrier_NK', 'Carrier_OO',
       'Carrier_UA', 'Carrier_WN', 'DEST_REGION', 'Dest_Region_Midwest',
       'Dest_Region_Mountain', 'Dest_Region_Northeast', 'Dest_Region_OCONUS',
       'Dest_Region_South', 'Dest_Region_Southwest', 'Dest_Region_West',
       'Flight_Status_Binary'],
      dtype='object')
['FL_DATE', 'CRS_DEP_TIME', 'OP_UNIQUE_CARRIER', 'DEST', 'wind_dir_degrees', 'wind_speed_kt', 'wind_gust_kt', 'visibility_statute_mi', 'temperature_c', 'dewpoint_c', 'altimeter_hpa', 'Flight_Status', 'Carrier_AA', 'Carrier_AS', 'Carrier_B6', 'Carrier_DL', 'Carrier_F9', 'Carrier_G4', 'Carrier_HA', 'Carrier_NK', 'Carrier_OO', 'Carrier_UA', 'Carrier_WN', 'DEST_RE

### Split Data

### Training, Validation, Test Sets

In [25]:
# Feature selection
features_to_use = [
    'wind_dir_degrees', 'wind_speed_kt', 'temperature_c', 'dewpoint_c',
    'visibility_statute_mi', 'altimeter_hpa', 'Carrier_AA', 'Carrier_AS',
    'Carrier_B6', 'Carrier_DL', 'Carrier_F9', 'Carrier_G4', 'Carrier_HA',
    'Carrier_NK', 'Carrier_OO', 'Carrier_UA', 'Carrier_WN', 'Dest_Region_Midwest',
    'Dest_Region_Mountain', 'Dest_Region_Northeast', 'Dest_Region_OCONUS',
    'Dest_Region_South', 'Dest_Region_Southwest', 'Dest_Region_West'
]

# target variable
target = 'Flight_Status_Binary'

split

In [26]:
from sklearn.model_selection import train_test_split

# Split into features (X) and target (y)
X = processed_data[features_to_use]
y = processed_data[target]

# Split into training and temp (validation + test) datasets
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Split temp into validation and test sets (50% for validation, 50% for test)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

# Check the sizes of the splits
print(f"Training set size: {len(X_train)}")
print(f"Validation set size: {len(X_val)}")
print(f"Test set size: {len(X_test)}")

Training set size: 129920
Validation set size: 27840
Test set size: 27840


Balancing the Dataset (Undersampling)

In [27]:
from sklearn.utils import resample

#  features and target
train_data = pd.concat([X_train, y_train], axis=1)

# Separate the majority and minority classes
majority_class = train_data[train_data[target] == 0]
minority_class = train_data[train_data[target] == 1]

# Undersample the majority class
majority_class_downsampled = resample(majority_class,
                                      replace=False,
                                      n_samples=len(minority_class),
                                      random_state=42)

# Combine the downsampled majority class with the minority class
balanced_train_data = pd.concat([majority_class_downsampled, minority_class])

# Separate back into X and y
X_train_balanced = balanced_train_data.drop(columns=[target])
y_train_balanced = balanced_train_data[target]

# Check
print(f"Class distribution in training set after balancing:\n{y_train_balanced.value_counts()}")

Class distribution in training set after balancing:
Flight_Status_Binary
0    13852
1    13852
Name: count, dtype: int64


Feature Scaling

In [28]:
from sklearn.preprocessing import StandardScaler

# StandardScaler
scaler = StandardScaler()

# transform the training data
X_train_scaled = scaler.fit_transform(X_train_balanced)

# Transform the validation and test data
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Check
print(f"Scaled Training data shape: {X_train_scaled.shape}")
print(f"Scaled Validation data shape: {X_val_scaled.shape}")
print(f"Scaled Test data shape: {X_test_scaled.shape}")

Scaled Training data shape: (27704, 24)
Scaled Validation data shape: (27840, 24)
Scaled Test data shape: (27840, 24)


## Model Training (RandomForest)

In [29]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# RandomForest model
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

# Train the model using the scaled training data
rf_model.fit(X_train_scaled, y_train_balanced)

# Make predictions on the validation set
val_preds = rf_model.predict(X_val_scaled)

# Evaluate performance on the validation set
print("Validation Set Performance:")
print(confusion_matrix(y_val, val_preds))
print(classification_report(y_val, val_preds))

Validation Set Performance:
[[15875  8996]
 [ 1034  1935]]
              precision    recall  f1-score   support

           0       0.94      0.64      0.76     24871
           1       0.18      0.65      0.28      2969

    accuracy                           0.64     27840
   macro avg       0.56      0.65      0.52     27840
weighted avg       0.86      0.64      0.71     27840



In [ ]:
# Make predictions on the test set
test_preds = rf_model.predict(X_test_scaled)

# Evaluate performance on the test set
print("\nTest Set Performance:")
print(confusion_matrix(y_test, test_preds))
print(classification_report(y_test, test_preds))

Validation Set:

The model was good at predicting on-time flights (94% precision), but it had trouble predicting delayed/canceled flights (only 18% precision).

It correctly identified 64% of all the delayed/canceled flights, but missed a lot of them.

Test Set:

The results were similar to the validation set: the model was good at predicting on-time flights (94% precision), but struggled with delayed/canceled flights (18% precision).

It correctly identified 64% of delayed/canceled flights, but still missed many.